[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.3_gqa_deep_dive/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.3_gqa_deep_dive/lab.ipynb)

# Lab 3.3: GQA Deep Dive

This lab explores Grouped-Query Attention through three experiments:
1. **Head sharing patterns**: how query heads map to KV heads across mechanisms
2. **Memory savings**: KV cache size as a function of group size
3. **Quality-memory tradeoff**: the Pareto frontier showing why GQA-8 dominates


In [ ]:
# ============================================================
# Setup: model configurations for real production LLMs
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters (change these to explore different models) ---
N_QUERY_HEADS = 32       # Llama 3.1 8B query heads
HEAD_DIM = 128           # standard head dimension
N_LAYERS = 32            # Llama 3.1 8B layers
DTYPE_BYTES = 2          # FP16
CONTEXT_LEN = 4096       # tokens
GPU_MEMORY_GB = 80       # A100-80GB
MODEL_MEMORY_GB = 16     # Llama 3.1 8B weights in FP16
GPU_COST_PER_HOUR = 2.0  # $/hour for A100


## Experiment 1: Head Sharing Patterns

Visualize how query heads map to KV heads for MHA, GQA-8, GQA-4, and MQA. Each row is a query head, each column is a KV head. A filled cell means that query head reads from that KV head.


In [ ]:
# ============================================================
# Visualize the head-to-KV mapping for each attention variant
# ============================================================
def head_sharing_matrix(n_query, n_kv):
    """Build binary matrix: matrix[q][k] = 1 if query head q uses KV head k."""
    # Each query head maps to kv_head = q_head // group_size
    group_size = n_query // n_kv
    matrix = np.zeros((n_query, n_kv))
    for q in range(n_query):
        kv_idx = q // group_size
        matrix[q, kv_idx] = 1
    return matrix

# Build matrices for each mechanism
mechanisms = {
    "MHA (32 KV)": head_sharing_matrix(32, 32),
    "GQA-8 (8 KV)": head_sharing_matrix(32, 8),
    "GQA-4 (4 KV)": head_sharing_matrix(32, 4),
    "MQA (1 KV)": head_sharing_matrix(32, 1),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle("Query-to-KV Head Mapping Across Attention Variants", fontsize=14, fontweight="bold")

for ax, (name, matrix) in zip(axes, mechanisms.items()):
    # Plot heatmap with green for active connections
    ax.imshow(matrix, cmap="Greens", aspect="auto", vmin=0, vmax=1)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("KV Head Index")
    ax.set_ylabel("Query Head Index")
    # Show group boundaries
    n_kv = matrix.shape[1]
    group_size = 32 // n_kv
    for i in range(1, n_kv):
        ax.axhline(i * group_size - 0.5, color="black", linewidth=0.5, linestyle="--")

plt.tight_layout()
plt.savefig("head_sharing_patterns.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: head_sharing_patterns.png")


## Experiment 2: KV Cache Memory Savings

Calculate exact KV cache size per token and total cache at context length for each attention variant. Then compute how many concurrent users fit on an A100-80GB.


In [ ]:
# ============================================================
# KV cache memory calculation for each attention variant
# ============================================================
def kv_cache_per_token_bytes(n_kv_heads, head_dim, n_layers, dtype_bytes):
    """KV cache bytes per token = 2 (K+V) * n_kv_heads * head_dim * layers * dtype."""
    return 2 * n_kv_heads * head_dim * n_layers * dtype_bytes

def max_concurrent_users(gpu_mem_gb, model_mem_gb, kv_per_token, context_len, overhead=1.2):
    """Max batch size given available memory after model weights."""
    available_bytes = (gpu_mem_gb - model_mem_gb) * (1024**3)
    kv_per_sequence = kv_per_token * context_len
    return int(available_bytes / (kv_per_sequence * overhead))

# Define variants: (label, n_kv_heads)
variants = [
    ("MHA (32)", 32),
    ("GQA-16", 16),
    ("GQA-8", 8),
    ("GQA-4", 4),
    ("GQA-2", 2),
    ("MQA (1)", 1),
]

# Compute metrics
labels = []
kv_per_token_kb = []
total_cache_mb = []
concurrent_users = []
cost_per_user_hr = []

for name, n_kv in variants:
    kv_bytes = kv_cache_per_token_bytes(n_kv, HEAD_DIM, N_LAYERS, DTYPE_BYTES)
    users = max_concurrent_users(GPU_MEMORY_GB, MODEL_MEMORY_GB, kv_bytes, CONTEXT_LEN)
    labels.append(name)
    kv_per_token_kb.append(kv_bytes / 1024)
    total_cache_mb.append(kv_bytes * CONTEXT_LEN / (1024**2))
    concurrent_users.append(users)
    cost_per_user_hr.append(GPU_COST_PER_HOUR / max(users, 1))  # $/user/hour

# Print table
print(f"{'Mechanism':<12} {'KB/token':>10} {'MB/seq (4K)':>12} {'Users':>8} {'$/user/hr':>10}")
print("-" * 56)
for i in range(len(labels)):
    print(f"{labels[i]:<12} {kv_per_token_kb[i]:>10.1f} {total_cache_mb[i]:>12.1f} {concurrent_users[i]:>8} {cost_per_user_hr[i]:>10.4f}")


In [ ]:
# ============================================================
# Chart 1: KV cache per token by mechanism
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: KV cache size bar chart
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(labels)))  # red=big, green=small
bars = ax1.bar(labels, kv_per_token_kb, color=colors, edgecolor="black", linewidth=0.8)
ax1.set_ylabel("KV Cache per Token (KB)")
ax1.set_title("KV Cache Size by Attention Mechanism\n(Llama 3.1 8B config)", fontweight="bold")
ax1.set_yscale("log")
ax1.set_ylim(1, 1000)

# Annotate compression ratio relative to MHA
mha_kb = kv_per_token_kb[0]
for bar, kb in zip(bars, kv_per_token_kb):
    ratio = mha_kb / kb
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.15,
             f"{ratio:.0f}x" if ratio > 1 else "1x",
             ha="center", va="bottom", fontsize=9, fontweight="bold")

# Right: concurrent users bar chart
colors2 = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(labels)))
bars2 = ax2.barh(labels, concurrent_users, color=colors2, edgecolor="black", linewidth=0.8)
ax2.set_xlabel("Max Concurrent Users (A100-80GB, 4K context)")
ax2.set_title("Serving Capacity by Attention Mechanism", fontweight="bold")

# Annotate cost per user
for bar, cost in zip(bars2, cost_per_user_hr):
    ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f"${cost:.4f}/hr" if cost < 0.01 else f"${cost:.3f}/hr",
             ha="left", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("kv_cache_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: kv_cache_comparison.png")


## Experiment 3: Quality-Memory Tradeoff (Pareto Frontier)

Plot benchmark quality vs KV memory to visualize the Pareto frontier. Data points from published results (Ainslie et al. 2023, Llama 3 paper).


In [ ]:
# ============================================================
# Quality vs Memory Pareto frontier
# Published benchmark scores (MMLU) for different mechanisms
# ============================================================
# Data points: (mechanism, kv_kb_per_token, mmlu_score, model_name)
data_points = [
    ("MHA", 524.0, 69.8, "Llama 2 7B (MHA)"),
    ("GQA-8", 131.0, 69.4, "Llama 2 7B (uptrained GQA)"),
    ("GQA-8", 131.0, 73.0, "Llama 3.1 8B"),
    ("GQA-8", 131.0, 72.2, "Mistral 7B"),
    ("MQA", 16.4, 67.5, "Falcon 7B (MQA)"),
    ("GQA-4", 65.5, 68.8, "GQA-4 (Ainslie ablation)"),
]

fig, ax = plt.subplots(figsize=(10, 6))

# Color map for mechanism types
mech_colors = {"MHA": "#ef4444", "GQA-8": "#22c55e", "GQA-4": "#3b82f6", "MQA": "#8b5cf6"}

for mech, kv_kb, score, name in data_points:
    color = mech_colors.get(mech, "#6b7280")
    ax.scatter(kv_kb, score, s=120, c=color, edgecolors="black", linewidth=1, zorder=5)
    # Offset labels to avoid overlap
    offset_x = 10 if kv_kb < 200 else -10
    ha = "left" if kv_kb < 200 else "right"
    ax.annotate(name, (kv_kb, score), textcoords="offset points",
                xytext=(offset_x, 8), ha=ha, fontsize=8)

# Highlight GQA sweet spot region
ax.axvspan(50, 200, alpha=0.1, color="green", label="GQA sweet spot")
ax.axvline(131, color="green", linestyle="--", alpha=0.5, linewidth=1)
ax.text(135, 66.5, "GQA-8\n(131 KB/token)", fontsize=8, color="green", style="italic")

ax.set_xlabel("KV Cache per Token (KB, log scale)", fontsize=11)
ax.set_ylabel("MMLU Score (%)", fontsize=11)
ax.set_title("Quality vs Memory: The GQA Sweet Spot", fontsize=13, fontweight="bold")
ax.set_xscale("log")
ax.set_xlim(10, 800)
ax.set_ylim(65, 75)
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("quality_memory_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: quality_memory_tradeoff.png")


## Experiment 4: Tensor Parallelism Compatibility

Not all group sizes work with all TP degrees. Visualize which combinations are valid.


In [ ]:
# ============================================================
# TP compatibility matrix: which KV head counts work with which TP degrees
# ============================================================
tp_degrees = [1, 2, 4, 8]
kv_head_options = [32, 16, 8, 4, 2, 1]

fig, ax = plt.subplots(figsize=(8, 5))

# Build compatibility matrix
compat = np.zeros((len(kv_head_options), len(tp_degrees)))
for i, n_kv in enumerate(kv_head_options):
    for j, tp in enumerate(tp_degrees):
        # Valid if n_kv_heads >= tp and divisible
        compat[i, j] = 1.0 if (n_kv >= tp and n_kv % tp == 0) else 0.0

# Plot as colored grid
cmap = plt.cm.colors.ListedColormap(["#fee2e2", "#dcfce7"])  # red=invalid, green=valid
ax.imshow(compat, cmap=cmap, aspect="auto")

# Add text annotations
for i in range(len(kv_head_options)):
    for j in range(len(tp_degrees)):
        text = "\u2713" if compat[i, j] == 1 else "\u2717"
        color = "#166534" if compat[i, j] == 1 else "#991b1b"
        ax.text(j, i, text, ha="center", va="center", fontsize=16, color=color, fontweight="bold")

ax.set_xticks(range(len(tp_degrees)))
ax.set_xticklabels([f"TP={tp}" for tp in tp_degrees])
ax.set_yticks(range(len(kv_head_options)))
ax.set_yticklabels([f"{n} KV heads" for n in kv_head_options])
ax.set_title("Tensor Parallelism Compatibility\n(GQA-8 works for all standard configs)", fontweight="bold")

# Highlight the GQA-8 row
ax.axhline(2 - 0.5, color="green", linewidth=2)
ax.axhline(2 + 0.5, color="green", linewidth=2)
ax.text(3.6, 2, "\u2190 Industry\n   default", fontsize=9, color="green", va="center")

plt.tight_layout()
plt.savefig("tp_compatibility.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tp_compatibility.png")


## Key Takeaways

1. GQA-8 delivers 4x KV cache compression with zero measurable quality loss
2. The choice of 8 KV heads is driven by tensor parallelism constraints (8-GPU nodes), not quality optimization
3. Memory savings translate directly to serving capacity: 4x smaller cache = 4x more concurrent users
4. The Pareto frontier shows GQA-8 as the clear optimal point for quality/memory tradeoff at 7B+ scale
